In [1]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /Users/msawant/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/msawant/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
import pandas as pd
import numpy as np
from nltk.tokenize import word_tokenize
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import TransformerMixin, BaseEstimator
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix,f1_score, precision_score, recall_score

In [4]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
#file_path = "EXIST_2023/disagreement_records_task1.xlsx"
#file_path="LLM_processed_dataset/LLM_EXIST2023_majority.xlsx"
#data = pd.read_csv("LLM_processed_dataset/All_LLM_EXIST2023.csv")


In [6]:
#len(data)

In [7]:
import re

def remove_urls_and_lower(text):
    # Define the regex pattern for URLs starting with http or https
    url_pattern = re.compile(r'http[s]?://\S+')
    # Substitute the URLs with an empty string
    cleaned_text = url_pattern.sub('', text)
    cleaned_text = cleaned_text.lower()
    return cleaned_text.strip()

In [8]:
def clean_text(text):
    return re.sub(r'[^A-Za-z\s]', '', text) if isinstance(text, str) else text
    #return re.sub(r'[^A-Za-z0-9\s]', '', text) if isinstance(text, str) else text

In [9]:
from nltk.corpus import stopwords
stop_words=set(stopwords.words('english'))
#additional_stopwords={'ago','also','along','always','amp','like','look','know','even','want','say','get','far','use','take','never','great','whole','know','even','use','take','would'}
stop_words.update(['ago','also','along','always','amp','like','look','know','even','want','say','get','far','use','take','never','great','whole','would','ok','dont','cant','shes','theyre','without','youre','isnt','us','yet','yall','u','id'])


#stop_words.update(['ago','also','along','always','amp','like','look','know','even','want','say','get','far','use','take','never','great','whole','would','ok','dont','cant','shes','theyre','without','youre'])

def stopwords_removal(text):
    words=word_tokenize(text)
    filtered_words=[w for w in words if w.lower() not in stop_words]
    return ' '.join(filtered_words)

In [10]:
from nltk.corpus import wordnet
def pos_tagger(nltk_tag):
    if nltk_tag.startswith('J'):
        return wordnet.ADJ
    elif nltk_tag.startswith('V'):
        return wordnet.VERB
    elif nltk_tag.startswith('N'):
        return wordnet.NOUN
    elif nltk_tag.startswith('R'):
        return wordnet.ADV
    else:          
        return None

In [11]:
from nltk.stem import WordNetLemmatizer
def apply_lemmatization(text):
    pos_tagged = nltk.pos_tag(nltk.word_tokenize(text))  
    wordnet_tagged = list(map(lambda x: (x[0], pos_tagger(x[1])), pos_tagged))
    lemmatizer = WordNetLemmatizer()
    lemmatized_sentence = []
    for word, tag in wordnet_tagged:
        if tag is None:
            # if there is no available tag, append the token as is
            lemmatized_sentence.append(word)
        else:        
        # else use the tag to lemmatize the token
            lemmatized_sentence.append(lemmatizer.lemmatize(word, tag))
    lemmatized_sentence = " ".join(lemmatized_sentence)
    return lemmatized_sentence

Function to execute random forest

In [12]:
def rf_model(tweet,Y):
    tweet = tweet.reset_index(drop=True)
    Y = Y.reset_index(drop=True)
    
    tweet_processed=tweet.apply(remove_urls_and_lower)
    tweet_processed=tweet_processed.apply(clean_text)
    tweet_processed=tweet_processed.apply(apply_lemmatization)
    tweet_processed=tweet_processed.apply(stopwords_removal)
    
    vec=TfidfVectorizer(tokenizer=word_tokenize,token_pattern=None,ngram_range=(1,1))
    tweet_vectorized=vec.fit_transform(tweet_processed)
    
    X_train, X_test, y_train, y_test = train_test_split(tweet_vectorized,Y, test_size=0.20, random_state=42)
    
    rf_model=RandomForestClassifier(bootstrap=False, min_samples_leaf=4,min_samples_split=10,n_estimators=100,n_jobs=-1,random_state=10)
    rf_model.fit(X_train, y_train)
    
    y_pred_rf =rf_model.predict(X_test)
    #y_pred_rf = y_pred_rf.astype(int)
    print("ACCURACY OF THE MODEL:", accuracy_score(y_test, y_pred_rf))
    print("F1 score:", f1_score(y_test, y_pred_rf, average='macro'))
    print(f"Precision: {precision_score(y_test, y_pred_rf, average='macro')}")
    print(f"Recall: {recall_score(y_test, y_pred_rf, average='macro')}")

    target_names = ['Yes', 'No']
    print("Classification Report for Human Annotators")
    print(classification_report(y_test, y_pred_rf, target_names=target_names))

**AGE persona**

In [13]:
df1=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_EXIST2023_age18.csv")
print("GPT-5.4; Age 18")
rf_model(df1["tweet"],df1["GPT_5.4"])

GPT-5.4; Age 18
ACCURACY OF THE MODEL: 0.7944785276073619
F1 score: 0.7888988759701161
Precision: 0.7926632586235547
Recall: 0.7867218248154213
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.80      0.85      0.82       369
          No       0.78      0.73      0.75       283

    accuracy                           0.79       652
   macro avg       0.79      0.79      0.79       652
weighted avg       0.79      0.79      0.79       652



In [14]:
df2=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_EXIST2023_age23.csv")
print("GPT-5.4; Age 23")
rf_model(df2["tweet"],df2["GPT_5.4"])

GPT-5.4; Age 23
ACCURACY OF THE MODEL: 0.7822085889570553
F1 score: 0.7758278370185276
Precision: 0.7800235478806907
Recall: 0.7735288479969331
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.79      0.84      0.81       370
          No       0.77      0.71      0.74       282

    accuracy                           0.78       652
   macro avg       0.78      0.77      0.78       652
weighted avg       0.78      0.78      0.78       652



In [15]:
df3=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_EXIST2023_age46.csv")
print("GPT-5.4; Age 46")
rf_model(df3["tweet"],df3["GPT_5.4"])

GPT-5.4; Age 46
ACCURACY OF THE MODEL: 0.7914110429447853
F1 score: 0.785748112924894
Precision: 0.7899393227637503
Recall: 0.7834698407838334
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.80      0.85      0.82       368
          No       0.78      0.72      0.75       284

    accuracy                           0.79       652
   macro avg       0.79      0.78      0.79       652
weighted avg       0.79      0.79      0.79       652



In [16]:
df4=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_mini_EXIST2023_age18.csv")
print("GPT-5.4_mini; Age 18")
rf_model(df4["tweet"],df4["GPT_5.4_mini"])

GPT-5.4_mini; Age 18
ACCURACY OF THE MODEL: 0.7714723926380368
F1 score: 0.7714460481584736
Precision: 0.7714568040654997
Recall: 0.7714389227135423
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.77      0.78      0.77       329
          No       0.77      0.77      0.77       323

    accuracy                           0.77       652
   macro avg       0.77      0.77      0.77       652
weighted avg       0.77      0.77      0.77       652



In [17]:
df5=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_mini_EXIST2023_age23.csv")
print("GPT-5.4_mini; Age 23")
rf_model(df5["tweet"],df5["GPT_5.4_mini"])

GPT-5.4_mini; Age 23
ACCURACY OF THE MODEL: 0.7852760736196319
F1 score: 0.7850313221233103
Precision: 0.7861649476746383
Recall: 0.7850986148750376
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.80      0.76      0.78       324
          No       0.77      0.81      0.79       328

    accuracy                           0.79       652
   macro avg       0.79      0.79      0.79       652
weighted avg       0.79      0.79      0.79       652



In [18]:
df6=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_mini_EXIST2023_age46.csv")
print("GPT-5.4_mini; Age 46")
rf_model(df6["tweet"],df6["GPT_5.4_mini"])

GPT-5.4_mini; Age 46
ACCURACY OF THE MODEL: 0.7852760736196319
F1 score: 0.7852033132530121
Precision: 0.785450572634117
Recall: 0.7851927130382415
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.79      0.77      0.78       324
          No       0.78      0.80      0.79       328

    accuracy                           0.79       652
   macro avg       0.79      0.79      0.79       652
weighted avg       0.79      0.79      0.79       652



In [19]:
df7=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_nano_EXIST2023_age18.csv")
print("GPT-5.4_nano; Age 18")
rf_model(df7["tweet"],df7["GPT_5.4_nano"])

GPT-5.4_nano; Age 18
ACCURACY OF THE MODEL: 0.8159509202453987
F1 score: 0.5582156973461322
Precision: 0.7506858054226475
Recall: 0.5582432893118117
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.82      0.98      0.90       523
          No       0.68      0.13      0.22       129

    accuracy                           0.82       652
   macro avg       0.75      0.56      0.56       652
weighted avg       0.79      0.82      0.76       652



In [20]:
df8=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_nano_EXIST2023_age23.csv")
print("GPT-5.4_nano; Age 23")
rf_model(df8["tweet"],df8["GPT_5.4_nano"])

GPT-5.4_nano; Age 23
ACCURACY OF THE MODEL: 0.8205521472392638
F1 score: 0.6247029710276834
Precision: 0.793247462919594
Recall: 0.6053613249751076
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.82      0.98      0.90       513
          No       0.76      0.23      0.35       139

    accuracy                           0.82       652
   macro avg       0.79      0.61      0.62       652
weighted avg       0.81      0.82      0.78       652



In [21]:
df9=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_nano_EXIST2023_age46.csv")
print("GPT-5.4_nano; Age 46")
rf_model(df9["tweet"],df9["GPT_5.4_nano"])

GPT-5.4_nano; Age 46
ACCURACY OF THE MODEL: 0.838957055214724
F1 score: 0.5876474946242388
Precision: 0.8162154989384289
Recall: 0.5751415012534093
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.84      0.99      0.91       533
          No       0.79      0.16      0.27       119

    accuracy                           0.84       652
   macro avg       0.82      0.58      0.59       652
weighted avg       0.83      0.84      0.79       652



In [22]:
df10=pd.read_csv("LLM_annotation5_age_persona/mistral_EXIST2023_age18.csv")
print("mistral; Age 18")
rf_model(df10["tweet"],df10["mistral"])

mistral; Age 18
ACCURACY OF THE MODEL: 0.7177914110429447
F1 score: 0.6663551817736839
Precision: 0.7172350922350923
Recall: 0.6617294793995528
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.72      0.43      0.54       248
          No       0.72      0.90      0.80       404

    accuracy                           0.72       652
   macro avg       0.72      0.66      0.67       652
weighted avg       0.72      0.72      0.70       652



In [23]:
df11=pd.read_csv("LLM_annotation5_age_persona/mistral_EXIST2023_age23.csv")
print("mistral; Age 23")
rf_model(df11["tweet"],df11["mistral"])

mistral; Age 23
ACCURACY OF THE MODEL: 0.7223926380368099
F1 score: 0.6533371716952145
Precision: 0.7171878515185601
Recall: 0.6480493191019507
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.71      0.38      0.50       234
          No       0.73      0.91      0.81       418

    accuracy                           0.72       652
   macro avg       0.72      0.65      0.65       652
weighted avg       0.72      0.72      0.70       652



In [24]:
df12=pd.read_csv("LLM_annotation5_age_persona/mistral_EXIST2023_age46.csv")
print("mistral; Age 46")
rf_model(df12["tweet"],df12["mistral"])

mistral; Age 46
ACCURACY OF THE MODEL: 0.7147239263803681
F1 score: 0.6627286076625283
Precision: 0.7079615603230625
Recall: 0.6577746577746577
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.70      0.43      0.53       245
          No       0.72      0.89      0.80       407

    accuracy                           0.71       652
   macro avg       0.71      0.66      0.66       652
weighted avg       0.71      0.71      0.70       652



In [25]:
df13=pd.read_csv("LLM_annotation5_age_persona/phi_EXIST2023_age18.csv")
print("phi; Age 18")
rf_model(df13["tweet"],df13["phi"])

phi; Age 18
ACCURACY OF THE MODEL: 0.745398773006135
F1 score: 0.6685165447707254
Precision: 0.7255620723362659
Recall: 0.6575555840054623
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.76      0.91      0.83       438
          No       0.69      0.40      0.51       214

    accuracy                           0.75       652
   macro avg       0.73      0.66      0.67       652
weighted avg       0.74      0.75      0.72       652



In [26]:
df14=pd.read_csv("LLM_annotation5_age_persona/phi_EXIST2023_age23.csv")
print("phi; Age 23")
rf_model(df14["tweet"],df14["phi"])

phi; Age 23
ACCURACY OF THE MODEL: 0.7300613496932515
F1 score: 0.6550762275765882
Precision: 0.7119922747299173
Recall: 0.6471820657867169
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.74      0.91      0.82       430
          No       0.68      0.39      0.49       222

    accuracy                           0.73       652
   macro avg       0.71      0.65      0.66       652
weighted avg       0.72      0.73      0.71       652



In [27]:
df15=pd.read_csv("LLM_annotation5_age_persona/phi_EXIST2023_age46.csv")
print("phi; Age 46")
rf_model(df15["tweet"],df15["phi"])

phi; Age 46
ACCURACY OF THE MODEL: 0.7469325153374233
F1 score: 0.6469256141387288
Precision: 0.7305697115823698
Recall: 0.6373917018590274
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.75      0.93      0.83       446
          No       0.71      0.34      0.46       206

    accuracy                           0.75       652
   macro avg       0.73      0.64      0.65       652
weighted avg       0.74      0.75      0.72       652



In [28]:
df16=pd.read_csv("LLM_annotation5_age_persona/GPT_5.5_EXIST2023_age18.csv")
print("GPT 5.5; Age 18")
rf_model(df16["tweet"],df16["GPT_5.5"])

GPT 5.5; Age 18
ACCURACY OF THE MODEL: 0.8052147239263804
F1 score: 0.8042824153408701
Precision: 0.8037632382316361
Recall: 0.8064922471408064
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.84      0.79      0.82       359
          No       0.76      0.82      0.79       293

    accuracy                           0.81       652
   macro avg       0.80      0.81      0.80       652
weighted avg       0.81      0.81      0.81       652



In [29]:
df17=pd.read_csv("LLM_annotation5_age_persona/GPT_5.5_EXIST2023_age23.csv")
print("GPT_5.5 ; Age 23")
rf_model(df17["tweet"],df17["GPT_5.5"])

GPT_5.5 ; Age 23
ACCURACY OF THE MODEL: 0.8128834355828221
F1 score: 0.8122456686965963
Precision: 0.8118595539481616
Recall: 0.8138816213551738
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.85      0.80      0.82       354
          No       0.78      0.83      0.80       298

    accuracy                           0.81       652
   macro avg       0.81      0.81      0.81       652
weighted avg       0.81      0.81      0.81       652



In [30]:
df18=pd.read_csv("LLM_annotation5_age_persona/GPT_5.5_EXIST2023_age46.csv")
print("GPT_5.5; Age 46")
rf_model(df18["tweet"],df18["GPT_5.5"])

GPT_5.5; Age 46
ACCURACY OF THE MODEL: 0.8190184049079755
F1 score: 0.8178598484848485
Precision: 0.8175780510879849
Recall: 0.8182042240169871
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.84      0.83      0.83       354
          No       0.80      0.81      0.80       298

    accuracy                           0.82       652
   macro avg       0.82      0.82      0.82       652
weighted avg       0.82      0.82      0.82       652



**PROMPT 4 male persona**

In [31]:
df19=pd.read_csv("LLM_annotation4_male_persona/GPT_5.5_EXIST2023.csv")
print("GPT-5.5")
rf_model(df19["tweet"],df19["GPT_5.5"])

GPT-5.5
ACCURACY OF THE MODEL: 0.808282208588957
F1 score: 0.8071019614061912
Precision: 0.8064627295472785
Recall: 0.8083931896781058
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.84      0.81      0.82       358
          No       0.78      0.81      0.79       294

    accuracy                           0.81       652
   macro avg       0.81      0.81      0.81       652
weighted avg       0.81      0.81      0.81       652



In [32]:
df20=pd.read_csv("LLM_annotation4_male_persona/GPT_5.4_EXIST2023.csv")
print("GPT-5.4")
rf_model(df20["tweet"],df20["GPT_5.4"])

GPT-5.4
ACCURACY OF THE MODEL: 0.7975460122699386
F1 score: 0.7911615401787881
Precision: 0.7942597328786587
Recall: 0.78909265944645
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.81      0.85      0.83       375
          No       0.78      0.73      0.75       277

    accuracy                           0.80       652
   macro avg       0.79      0.79      0.79       652
weighted avg       0.80      0.80      0.80       652



In [33]:
df21=pd.read_csv("LLM_annotation4_male_persona/GPT_5.4_mini_EXIST2023.csv")
print("GPT-5.4-mini")
rf_model(df21["tweet"],df21["GPT_5.4_mini"])

GPT-5.4-mini
ACCURACY OF THE MODEL: 0.7883435582822086
F1 score: 0.788144213381555
Precision: 0.7882825161120115
Recall: 0.7880653786766091
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.79      0.78      0.78       318
          No       0.79      0.80      0.79       334

    accuracy                           0.79       652
   macro avg       0.79      0.79      0.79       652
weighted avg       0.79      0.79      0.79       652



In [34]:
df22=pd.read_csv("LLM_annotation4_male_persona/GPT_5.4_nano_EXIST2023.csv")
print("GPT-5.4-nano")
rf_model(df22["tweet"],df22["GPT_5.4_nano"])

GPT-5.4-nano
ACCURACY OF THE MODEL: 0.8174846625766872
F1 score: 0.5913818800393935
Precision: 0.8223003265622406
Recall: 0.5820927085095584
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.82      0.99      0.90       514
          No       0.83      0.17      0.29       138

    accuracy                           0.82       652
   macro avg       0.82      0.58      0.59       652
weighted avg       0.82      0.82      0.77       652



In [35]:
df23=pd.read_csv("LLM_annotation4_male_persona/mistral_EXIST2023.csv")
print("Mistral")
rf_model(df23["tweet"],df23["mistral"])

Mistral
ACCURACY OF THE MODEL: 0.7147239263803681
F1 score: 0.688249992287997
Precision: 0.7127082982662851
Recall: 0.6844009922661608
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.71      0.52      0.60       267
          No       0.72      0.85      0.78       385

    accuracy                           0.71       652
   macro avg       0.71      0.68      0.69       652
weighted avg       0.71      0.71      0.70       652



In [36]:
df24=pd.read_csv("LLM_annotation4_male_persona/phi_EXIST2023.csv")
print("Phi")
rf_model(df24["tweet"],df24["phi"])

Phi
ACCURACY OF THE MODEL: 0.75
F1 score: 0.5769520132157713
Precision: 0.7429688293227739
Recall: 0.5828256956943099
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.75      0.97      0.85       466
          No       0.73      0.19      0.31       186

    accuracy                           0.75       652
   macro avg       0.74      0.58      0.58       652
weighted avg       0.75      0.75      0.69       652



**PROMPT 3 Female PERSONA**

In [37]:
df25=pd.read_csv("LLM_annotation3_female_persona/GPT_5.5_EXIST2023.csv")
print("GPT-5.5")
rf_model(df25["tweet"],df25["GPT_5.5"])

GPT-5.5
ACCURACY OF THE MODEL: 0.8205521472392638
F1 score: 0.8196147979995034
Precision: 0.818979897650485
Recall: 0.8219748858447489
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.86      0.81      0.83       360
          No       0.78      0.84      0.81       292

    accuracy                           0.82       652
   macro avg       0.82      0.82      0.82       652
weighted avg       0.82      0.82      0.82       652



In [38]:
df26=pd.read_csv("LLM_annotation3_female_persona/GPT_5.4_EXIST2023.csv")
print("GPT-5.4")
rf_model(df26["tweet"],df26["GPT_5.4"])

GPT-5.4
ACCURACY OF THE MODEL: 0.7944785276073619
F1 score: 0.7899242103643289
Precision: 0.7889783400421698
Recall: 0.791121152435021
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.83      0.81      0.82       378
          No       0.75      0.77      0.76       274

    accuracy                           0.79       652
   macro avg       0.79      0.79      0.79       652
weighted avg       0.80      0.79      0.79       652



In [39]:
df27=pd.read_csv("LLM_annotation3_female_persona/GPT_5.4_mini_EXIST2023.csv")
print("GPT-5.4-mini")
rf_model(df27["tweet"],df27["GPT_5.4_mini"])

GPT-5.4-mini
ACCURACY OF THE MODEL: 0.7745398773006135
F1 score: 0.7743482529435447
Precision: 0.774265813253012
Recall: 0.7746095922040959
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.79      0.77      0.78       339
          No       0.76      0.78      0.77       313

    accuracy                           0.77       652
   macro avg       0.77      0.77      0.77       652
weighted avg       0.77      0.77      0.77       652



In [40]:
df28=pd.read_csv("LLM_annotation3_female_persona/GPT_5.4_nano_EXIST2023.csv")
print("GPT-5.4-nano")
rf_model(df28["tweet"],df28["GPT_5.4_nano"])

GPT-5.4-nano
ACCURACY OF THE MODEL: 0.8450920245398773
F1 score: 0.6128152211619307
Precision: 0.7960885148823438
Recall: 0.5916752444673186
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.85      0.99      0.91       536
          No       0.74      0.20      0.31       116

    accuracy                           0.85       652
   macro avg       0.80      0.59      0.61       652
weighted avg       0.83      0.85      0.81       652



In [41]:
df29=pd.read_csv("LLM_annotation3_female_persona/mistral_EXIST2023.csv")
print("Mistral")
rf_model(df29["tweet"],df29["mistral"])

Mistral
ACCURACY OF THE MODEL: 0.7116564417177914
F1 score: 0.6619562939184345
Precision: 0.7105920609112899
Recall: 0.6582845674657977
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.71      0.43      0.53       251
          No       0.71      0.89      0.79       401

    accuracy                           0.71       652
   macro avg       0.71      0.66      0.66       652
weighted avg       0.71      0.71      0.69       652



In [42]:
df30=pd.read_csv("LLM_annotation3_female_persona/phi_EXIST2023.csv")
print("Phi")
rf_model(df30["tweet"],df30["phi"])

Phi
ACCURACY OF THE MODEL: 0.75920245398773
F1 score: 0.6079194420079745
Precision: 0.7391462527285957
Recall: 0.6030518394648829
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.76      0.96      0.85       468
          No       0.71      0.24      0.36       184

    accuracy                           0.76       652
   macro avg       0.74      0.60      0.61       652
weighted avg       0.75      0.76      0.71       652



**PROMPT2**

In [43]:
print("GPT-5.4")
df31 = pd.read_csv("LLM_annotation2/GPT_5.4_EXIST2023.csv")
rf_model(df31["tweet"],df31["GPT_5.4"])

GPT-5.4
ACCURACY OF THE MODEL: 0.7837423312883436
F1 score: 0.7731323246706134
Precision: 0.7797923805684338
Recall: 0.7694536383666417
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.80      0.85      0.82       383
          No       0.76      0.69      0.72       269

    accuracy                           0.78       652
   macro avg       0.78      0.77      0.77       652
weighted avg       0.78      0.78      0.78       652



In [44]:
print("GPT-5.4-mini")
df32 = pd.read_csv("LLM_annotation2/GPT_5.4_mini_EXIST2023.csv")
rf_model(df32["tweet"],df32["GPT_5.4_mini"])

GPT-5.4-mini
ACCURACY OF THE MODEL: 0.7822085889570553
F1 score: 0.7820034659433394
Precision: 0.7819577192899854
Recall: 0.7820639630728652
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.79      0.79      0.79       337
          No       0.77      0.78      0.78       315

    accuracy                           0.78       652
   macro avg       0.78      0.78      0.78       652
weighted avg       0.78      0.78      0.78       652



In [45]:
print("GPT-5.4-nano")
df33 = pd.read_csv("LLM_annotation2/GPT_5.4_nano_EXIST2023.csv")
rf_model(df33["tweet"],df33["GPT_5.4_nano"])

GPT-5.4-nano
ACCURACY OF THE MODEL: 0.7944785276073619
F1 score: 0.6043115942028986
Precision: 0.7661724327292696
Recall: 0.5943791782955798
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.80      0.98      0.88       497
          No       0.73      0.21      0.33       155

    accuracy                           0.79       652
   macro avg       0.77      0.59      0.60       652
weighted avg       0.78      0.79      0.75       652



In [46]:
print("GPT-5.5")
df34 = pd.read_csv("LLM_annotation2/GPT_5.5_EXIST2023.csv")
rf_model(df34["tweet"],df34["GPT_5.5"])

GPT-5.5
ACCURACY OF THE MODEL: 0.8098159509202454
F1 score: 0.8092973666581751
Precision: 0.8090692573451195
Recall: 0.8110604754280084
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.84      0.80      0.82       353
          No       0.77      0.83      0.80       299

    accuracy                           0.81       652
   macro avg       0.81      0.81      0.81       652
weighted avg       0.81      0.81      0.81       652



In [47]:
print("phi")
df35 = pd.read_csv("LLM_annotation2/phi_EXIST2023.csv")
rf_model(df35["tweet"],df35["phi"])

phi
ACCURACY OF THE MODEL: 0.7239263803680982
F1 score: 0.6971104710582552
Precision: 0.7099060956413179
Recall: 0.6921044003536974
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.75      0.83      0.79       401
          No       0.67      0.55      0.61       251

    accuracy                           0.72       652
   macro avg       0.71      0.69      0.70       652
weighted avg       0.72      0.72      0.72       652



In [48]:
print("mistral")
df36 = pd.read_csv("LLM_annotation2/mistral_EXIST2023_Second_time.csv")
rf_model(df36["tweet"],df36["mistral"])

mistral
ACCURACY OF THE MODEL: 0.7346625766871165
F1 score: 0.6910367345149954
Precision: 0.7366499256905652
Recall: 0.6840712726837873
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.74      0.47      0.57       249
          No       0.73      0.90      0.81       403

    accuracy                           0.73       652
   macro avg       0.74      0.68      0.69       652
weighted avg       0.74      0.73      0.72       652



In [49]:
print("llama")
df37 = pd.read_csv("LLM_annotation2/llama_EXIST2023_Second_time.csv")
#df2 = df2[df2["llama"].astype(str).str.strip().str.upper().isin(["YES", "NO"])]
rf_model(df37["tweet"],df37["llama"])

llama
ACCURACY OF THE MODEL: 0.7561349693251533
F1 score: 0.6597646827155024
Precision: 0.7612302749692903
Recall: 0.6489226459814694
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.75      0.95      0.84       442
          No       0.77      0.35      0.48       210

    accuracy                           0.76       652
   macro avg       0.76      0.65      0.66       652
weighted avg       0.76      0.76      0.72       652



**Individual human annotations**

In [50]:
#prompt2
data=pd.read_csv("LLM_annotation2/All_LLM_Prompt2_Combined.csv")

In [51]:
print("anno1")
rf_model(data["tweet"],data["anno1"])

anno1
ACCURACY OF THE MODEL: 0.7055214723926381
F1 score: 0.657563025210084
Precision: 0.6816938395885764
Recall: 0.6519444868079914
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.73      0.85      0.79       415
          No       0.63      0.46      0.53       237

    accuracy                           0.71       652
   macro avg       0.68      0.65      0.66       652
weighted avg       0.70      0.71      0.69       652



In [52]:
print("anno2")
rf_model(data["tweet"],data["anno2"])

anno2
ACCURACY OF THE MODEL: 0.7101226993865031
F1 score: 0.6971384613115938
Precision: 0.7050817064645871
Recall: 0.694696649097834
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.72      0.80      0.76       374
          No       0.69      0.59      0.63       278

    accuracy                           0.71       652
   macro avg       0.71      0.69      0.70       652
weighted avg       0.71      0.71      0.71       652



In [53]:
print("anno3")
rf_model(data["tweet"],data["anno3"])

anno3
ACCURACY OF THE MODEL: 0.6641104294478528
F1 score: 0.6149346709275515
Precision: 0.6395533317061264
Recall: 0.6139545776156736
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.69      0.83      0.75       403
          No       0.59      0.40      0.48       249

    accuracy                           0.66       652
   macro avg       0.64      0.61      0.61       652
weighted avg       0.65      0.66      0.65       652



In [54]:
print("anno4")
rf_model(data["tweet"],data["anno4"])

anno4
ACCURACY OF THE MODEL: 0.696319018404908
F1 score: 0.6811972144021337
Precision: 0.6876344740794869
Recall: 0.6788699690402477
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.72      0.78      0.75       380
          No       0.66      0.57      0.61       272

    accuracy                           0.70       652
   macro avg       0.69      0.68      0.68       652
weighted avg       0.69      0.70      0.69       652



In [55]:
print("anno5")
rf_model(data["tweet"],data["anno5"])

anno5
ACCURACY OF THE MODEL: 0.6625766871165644
F1 score: 0.6409547839320757
Precision: 0.6542643560992185
Recall: 0.6399938328707987
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.68      0.79      0.73       376
          No       0.63      0.49      0.55       276

    accuracy                           0.66       652
   macro avg       0.65      0.64      0.64       652
weighted avg       0.66      0.66      0.65       652



In [56]:
print("anno6")
rf_model(data["tweet"],data["anno6"])

anno6
ACCURACY OF THE MODEL: 0.6579754601226994
F1 score: 0.6508722430994947
Precision: 0.6514222941720629
Recall: 0.6504783245712316
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.69      0.71      0.70       369
          No       0.61      0.59      0.60       283

    accuracy                           0.66       652
   macro avg       0.65      0.65      0.65       652
weighted avg       0.66      0.66      0.66       652

